In [ ]:
def GB_diff_processing_final(root_path, dataset_key):
    """
    Processes all 5 pixel files for a given dataset using the fixed file structure:
    ROOT_PATH/parent/type/GB/1.txt
    """
    if not root_path:
        return [np.nan] * 5

    config = DATASET_MAP[dataset_key]
    parent_folder = config['parent']
    type_folder = config['type']
    colour1 = config['colour1']
    colour2 = config['colour2']

    print(f"\n--- Processing: {dataset_key} ---")

    # 1. Define the base folder containing GB/GI subfolders
    # This path is: ROOT_PATH/unpassivated/Dark_CPD
    base_data_folder = os.path.join(root_path, parent_folder, type_folder)
    
    # 2. Define the exact GB and GI folders
    GB_folder = os.path.join(base_data_folder, 'GB')
    GI_folder = os.path.join(base_data_folder, 'GI')
    
    # 3. Define the save path for intermediate plots and text files
    # This will be: ROOT_PATH/Plots/unpassivated/Dark_CPD
    save_path = os.path.join(root_path, 'Plots', parent_folder, type_folder)
    os.makedirs(save_path, exist_ok=True) # Create folder if it doesn't exist

    y_peaks = []
    GB_peaks = []
    GI_peaks = []
    # Check if the necessary folders exist before starting the loop
    if not os.path.isdir(GB_folder) or not os.path.isdir(GI_folder):
        print(f"Error: Could not find 'GB' and/or 'GI' folders at {base_data_folder}. Skipping.")
        return [np.nan] * 5

    # 4. Iterate through the 5 pixel files (1 to 5)
    for pixel_index in range(1, 6):
        
        # Construct the file paths: e.g., .../GB/1.txt and .../GI/1.txt
        file_name = f'{pixel_index}.txt'
        GB_file_path = os.path.join(GB_folder, file_name)
        GI_file_path = os.path.join(GI_folder, file_name)
        #print(GB_file_path)
        #print(GI_file_path)
        if os.path.exists(GB_file_path) and os.path.exists(GI_file_path):
            print(f"  Processing Pixel {pixel_index}...")
            if 'CPD' in type_folder:
                measurement_type = 'CPD'
                print(f"Folder is a CPD measurement: {type_folder}")
            elif 'SPV' in type_folder:
                measurement_type = 'SPV'
                print(f"Folder is an SPV measurement: {type_folder}")
            else:
                measurement_type = 'Unknown'
                print(f"Folder type is neither CPD nor SPV: {type_folder}")
            peak_diff, GB_peak, GI_peak = grain_boundaries(
                GB_file_path, 
                GI_file_path, 
                pixel_index, 
                save_path, 
                colour1, 
                colour2,
                datatype = f"{measurement_type}"
            )
            y_peaks.append(peak_diff)
            GB_peaks.append(GB_peak)
            GI_peaks.append(GI_peak)
        else:
            print(f"  File(s) not found for Pixel {pixel_index}. Skipping.")
            print(f"{GB_file_path}")
            y_peaks.append(np.nan) 
            GB_peaks.append(np.nan)
            GI_peaks.append(np.nan)

    # Convert peak differences to mV and round
    y_peaks_adjusted = [round(x * 1000, 4) if not np.isnan(x) else np.nan for x in y_peaks]
    GB_peaks_adjusted = [round(x * 1000, 4) if not np.isnan(x) else np.nan for x in GB_peaks]
    GI_peaks_adjusted = [round(x * 1000, 4) if not np.isnan(x) else np.nan for x in GI_peaks]
    return y_peaks_adjusted, GB_peaks_adjusted, GI_peaks_adjusted